

# Propagação de Covariâncias com um LiDAR SLAM

### 1. Introdução Teórica: A Lei de Propagação em Forma Matricial

Nesta seção, relembramos que, se temos um conjunto de observações $L$ com uma matriz de variância-covariância $\Sigma_L$, e funções lineares (ou linearizadas) $Y = AX$, a propagação é dada por:

$$\Sigma_Y = A \Sigma_L A^T$$

---

### 2. Contextualização: LiDAR SLAM e Georreferenciamento

Aqui, explicamos o problema:

* **Entrada:** Coordenadas dos Pontos de Controle (com incerteza conhecida $\sigma_{PC}$).
* **Transformação:** Rotação e Translação da nuvem de pontos (Parâmetros de corpo rígido).
* **Saída:** Coordenadas dos pontos da nuvem georreferenciada e suas respectivas incertezas.

---

### 3. Atividade Prática no Google Colab


#### Passo 1: Configuração do Ambiente

In [1]:
import numpy as np

# Definindo a precisão do equipamento (ex: FJD Trion S1)
# Supondo erro nominal de 2cm em X, Y e 3cm em Z
sigma_x = 0.02
sigma_y = 0.02
sigma_z = 0.03

# Matriz de Variância-Covariância das Observações (Simplificada como diagonal)
Sigma_L = np.diag([sigma_x**2, sigma_y**2, sigma_z**2])

print("Matriz ΣL (Observações):")
print(Sigma_L)

Matriz ΣL (Observações):
[[0.0004 0.     0.    ]
 [0.     0.0004 0.    ]
 [0.     0.     0.0009]]


#### Passo 2: Definindo a Matriz de Design (Jacobiano)

Vamos simular uma translação simples onde a coordenada final $P_{geo} = P_{lidar} + T$. No caso de uma transformação linear simples, a matriz $A$ representa as derivadas parciais.

Para o registro de duas nuvens (análogo a poligonais), os alunos devem montar a matriz $A$ baseada nos parâmetros de transformação.

In [2]:
# Exemplo: Propagação para um ponto transformado
# Supondo uma matriz A de sensibilidade (Jacobiano)
A = np.array([[1.2, 0.1, 0.0],
              [0.1, 1.1, 0.0],
              [0.0, 0.0, 1.0]])

# Aplicação da Lei de Propagação: ΣY = A * ΣL * A.T
Sigma_Y = A @ Sigma_L @ A.T

print("Matriz ΣY (Resultante na Nuvem de Pontos):")
print(Sigma_Y)

Matriz ΣY (Resultante na Nuvem de Pontos):
[[5.80e-04 9.20e-05 0.00e+00]
 [9.20e-05 4.88e-04 0.00e+00]
 [0.00e+00 0.00e+00 9.00e-04]]


# 4.Propagação de covariancia do LiDAR
## 1. O Desenho do Problema (Setup)

Imagine o seguinte cenário para os alunos:

1. **Espaço do Objeto (Nacional/Referência):** Temos 5 pontos de controle (GCPs) com coordenadas conhecidas.
2. **Espaço da Nuvem (Local SLAM):** O LiDAR gerou uma nuvem em um sistema arbitrário.
3. **O Problema:** A nuvem precisa ser ajustada aos pontos de controle. Mas, antes de ajustar, queremos **estimar a precisão teórica** de um ponto qualquer da nuvem georreferenciada, considerando que:
* Os pontos de controle têm incertezas diferentes (ex: alguns via GNSS RTK, outros via Estação Total).
* O equipamento LiDAR tem sua própria incerteza nominal ($2\text{ cm}$).



### A Equação de Transformação Afim 3D (Simplificada)

Para cada ponto $i$:

$X_i = a_1x_i + b_1y_i + c_1z_i + d_1$

$Y_i = a_2x_i + b_2y_i + c_2z_i + d_2$

$Z_i = a_3x_i + b_3y_i + c_3z_i + d_3$

---

## 2. Estrutura do Notebook no Colab

### Passo 1: Definindo as incertezas dos pontos de controle

Em vez de uma matriz identidade, vamos criar uma MVC para os 5 pontos de controle.

In [11]:
import numpy as np

# Incertezas dos 5 pontos de controle (em metros)
# [sigma_N, sigma_E, sigma_H]
precisoes_controle = np.array([
    [0.01, 0.01, 0.02], # P1 (Excelente)
    [0.02, 0.02, 0.04], # P2
    [0.05, 0.05, 0.08], # P3 (Pior precisão)
    [0.02, 0.02, 0.03], # P4
    [0.01, 0.01, 0.02]  # P5
])

# Construindo a MVC das observações de controle (diagonal para simplificar em aula)
sigma_L_control = np.diag(precisoes_controle.flatten()**2)

### Passo 2: A Incerteza Nominal do LiDAR SLAM

O SLAM não é perfeito. Cada ponto da nuvem tem um erro intrínseco.

In [4]:
sigma_slam = 0.02 # 2 cm nominal
sigma_L_slam = np.eye(3) * (sigma_slam**2)

### Passo 3: Calculando a propagação para os parâmetros da matriz de transformação afim

In [10]:
import numpy as np

# Coordenadas locais (SLAM) dos 5 pontos de apoio (Exemplo)
# x_local, y_local, z_local
GCP_local = np.array([
    [10.2, 5.1, 1.0],
    [20.5, 4.8, 1.2],
    [15.0, 15.3, 0.9],
    [5.1, 12.0, 1.1],
    [12.3, 10.1, 3.5]
])

num_pontos = len(GCP_local)
A = np.zeros((3 * num_pontos, 12))

for i in range(num_pontos):
    x, y, z = GCP_local[i]
    # Preenchendo as linhas correspondentes ao ponto i
    A[3*i,     0:4]  = [x, y, z, 1] # Equação para X
    A[3*i + 1, 4:8]  = [x, y, z, 1] # Equação para Y
    A[3*i + 2, 8:12] = [x, y, z, 1] # Equação para Z

print("Dimensões da Matriz A:", A.shape)

Dimensões da Matriz A: (15, 12)


In [12]:
# Supondo que você já tenha a Sigma_L_control (15x15) definida
P = np.linalg.inv(sigma_L_control)

# MVC dos parâmetros
Sigma_X = np.linalg.inv(A.T @ P @ A)

print("MVC dos Parâmetros (Sigma_X) calculada!")

MVC dos Parâmetros (Sigma_X) calculada!


### Passo 4: Montando a Matriz A (Propagação)

Aqui entra o desafio para os alunos. Se queremos a precisão de um ponto transformado $P_{geo}$, precisamos das derivadas parciais em relação aos parâmetros calculados.

Para fins didáticos calculem a propagação para um ponto específico da nuvem $P_{alvo}(x,y,z)$, onde a matriz $A$ será composta pelas derivadas em relação as coordenadas locais desse ponto.

A Lógica Matemática: A Soma das Incertezas

Para um ponto georreferenciado $P_{geo}$, a função de transformação é $f(X, L)$, onde:

* $X$: São os **parâmetros** da transformação (calculados via Mínimos Quadrados usando os 5 GCPs).
* $L$: São as **coordenadas locais** medidas pelo LiDAR SLAM.

Pela Lei de Propagação de Covariâncias para funções com múltiplas variáveis independentes, a MVC do ponto final ($\Sigma_{P_{geo}}$) é a soma das contribuições:

$$\Sigma_{P_{geo}} = J_X \Sigma_X J_X^T + J_L \Sigma_L J_L^T$$

Onde:

1. **$J_X \Sigma_X J_X^T$**: É a incerteza que vem do **Ajustamento**. Se os pontos de controle (GCPs) forem ruins, os parâmetros da transformação serão incertos, e a nuvem toda "balançará" ou "esticará" de forma imprecisa.
2. **$J_L \Sigma_L J_L^T$**: É a incerteza que vem do **Equipamento** ($\sigma_{L\_slam} = 2\text{ cm}$). Mesmo que a transformação fosse perfeita, o ponto ainda teria o erro intrínseco do laser.

---

### Passo 6 Caluclo da incerteza


In [13]:
# --- PASSO 2: A incerteza do Sensor (Sigma_L_slam) ---
sigma_slam = 0.02 # 2 cm nominal do FJD Trion
Sigma_L_slam = np.eye(3) * (sigma_slam**2)

# --- PASSO 3: O Ponto que queremos transformar ---
# Ponto local medido pelo SLAM
x, y, z = 10.0, 5.0, 2.0

# --- PASSO 4: Jacobianos ---
# J_L: Derivadas em relação às coordenadas (x,y,z)
# Na afim, se os parâmetros de escala/rotação são aprox. 1 e 0:
J_L = np.eye(3)

# J_X: Derivadas em relação aos parâmetros (a1, b1, c1, d1...)
# Para X_geo = a1*x + b1*y + c1*z + d1, a derivada em relação a a1 é 'x'
J_X = np.zeros((3, 12))
J_X[0, 0:4] = [x, y, z, 1] # Derivadas de X_geo
J_X[1, 4:8] = [x, y, z, 1] # Derivadas de Y_geo
J_X[2, 8:12] = [x, y, z, 1] # Derivadas de Z_geo

# --- PASSO 5: A GRANDE PROPAGAÇÃO ---
# Incerteza vinda dos parâmetros + Incerteza vinda do sensor
Sigma_P_final = (J_X @ Sigma_X @ J_X.T) + (J_L @ Sigma_L_slam @ J_L.T)

print("Matriz de Variância-Covariância do Ponto Georreferenciado:")
print(Sigma_P_final)
print(f"\nPrecisão esperada em X: {np.sqrt(Sigma_P_final[0,0])*100:.2f} cm")

Matriz de Variância-Covariância do Ponto Georreferenciado:
[[0.00055877 0.         0.        ]
 [0.         0.00055877 0.        ]
 [0.         0.         0.00095687]]

Precisão esperada em X: 2.36 cm


Desafio Prático:
"Se o Ponto de Controle 3 (P3) tiver um erro de 10cm em vez de 5cm, o quanto isso degrada a precisão de um ponto no extremo oposto da nuvem?"